# SigAlg's `Operators.expectation` method

In [ ]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `Operators.expectation` method in SigAlg is a method for computing *expectations* of random variables and vectors, both unconditional and conditional versions. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/core/#sigalg.core.Operators.expectation).

## Mathematical definition

Let $X:\Omega \to \mathbb{R}$ be a random variable on a probability space $(\Omega, \mathcal{F},P)$ for which $E(X^2) < \infty$, and let $\mathcal{G}$ be a sub-$\sigma$-algebra of $\mathcal{F}$. The *conditional expectation* of $X$ with respect to $\mathcal{G}$ is any $\mathcal{G}$-measurable random variable $E(X \mid \mathcal{G})$ such that

$$
\int_B X \, dP = \int_B E(X\mid \mathcal{G}) \, dP,
$$

for all $B\in \mathcal{G}$. Using the inner product

$$
\langle X, Y \rangle \stackrel{\text{def}}{=} \int_\Omega XY \, dP
$$

in the Hilbert space $L^2(\Omega, \mathcal{F}, P)$, this equality may be rewritten as

$$
\langle X - E(X\mid \mathcal{G}), I_B \rangle = 0, \tag{$\ast$}
$$

where $I_B$ is the indicator function of $B$. In the case that $\Omega$ is finite (as it always is, in SigAlg), the $\sigma$-algebra $\mathcal{G}$ is determined by its (finitely many) atoms, and the subspace $L^2(\Omega, \mathcal{G}, P)$ of $L^2(\Omega, \mathcal{F},P)$ has an orthogonal basis given by the indicator functions of the atoms of $\mathcal{G}$ with nonzero probability. Then the equation ($\ast$) shows that $E(X \mid \mathcal{G})$ is the orthogonal projection of $X$ onto $L^2(\Omega, \mathcal{G},P)$. In particular, we have the generalized Fourier expansion

$$
E(X\mid \mathcal{G}) = \sum_B \frac{\langle X, I_B \rangle}{\|I_B\|^2} I_B = \sum_B \frac{\int_B X \, dP}{P(B)} I_B,
$$

where the sum extends over all atoms $B$ of $\mathcal{G}$ with nonzero probability. If for each such $B$ we define the conditional probability measure $P_B$ on $B$ with $P_B(C) = P(C)/P(B)$ for $C\subset B$, then $\int_B X \, dP/P(B)$ is the same as the (unconditional) expectation $E(X|_B) = \int_B X|_B \, dP_B$, where $X|_B : B \to \mathbb{R}$ is the restricted random variable. Thus, we have

$$
E(X\mid \mathcal{G}) = \sum_B E(X|_B) I_B,
$$

which shows that $E(X\mid \mathcal{G})$ takes the constant value $E(X|_B)$ on each atom $B$.

If $\mathcal{G} = \{\emptyset, \Omega\}$ is the trivial $\sigma$-algebra, then $\mathcal{G}$ has only one atom, namely $\Omega$ itself. Then from above we get

$$
E( X \mid \{\emptyset,\Omega\}) = \frac{\int_\Omega X \, dP}{P(\Omega)} I_\Omega = E(X),
$$

which shows $E(X \mid \{\emptyset, \Omega\})$ is the constant random variable equal to the unconditional expectation $E(X) = \int_\Omega X \, dP$ everywhere. At the other extreme, if $\mathcal{G}$ is equal to $\mathcal{F}$, then $X$ is already $\mathcal{G}$-measurable, hence it is in $L^2(\Omega, \mathcal{G}, P)$, and hence $E(X\mid \mathcal{G}) = X$.

Finally, if $X : \Omega \to \mathbb{R}^d$ is a random vector of dimension $d>1$, with components

$$
X = (X_1,X_2,\ldots,X_d),
$$

then this method returns a `RandomVector` whose component random variables are the conditional expectations $E(X_j \mid \mathcal{G})$, for $j=1,2,\ldots,d$.

## API examples


### Unconditional expectations

We begin by defining a sample space $\Omega = \{0,1,2,3,4\}$ and a probability measure $P$ on $\Omega$.

In [2]:
from sigalg.core import ProbabilityMeasure, SampleSpace

Omega = SampleSpace().from_sequence(size=5)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.15,
        2: 0.3,
        3: 0.25,
        4: 0.2,
    }
)

Define a random variable $X: \Omega \to \mathbb{R}$ on the sample space $\Omega$ and set its `probability_measure` attribute to $P$ so that all expectations will be computed relative to $P$.

In [3]:
from sigalg.core import RandomVariable

X = RandomVariable(domain=Omega).from_dict(
    {
        0: 1,
        1: 1,
        2: -3,
        3: 2,
        4: -3,
    }
)
X.prob_measure = P

All expectations in SigAlg are instances of `RandomVariable`. The unconditional expectation `E(X)` is thus a constant random variable whose value is the usual expectation $E(X) = \int_\Omega X \, dP$. The `item` method extracts $E(X)$ from `E(X)`.

In [4]:
from sigalg.core import Operators

E = Operators.expectation

expectation_rv = E(X)
expectation = E(X).item()

print(expectation_rv, "\n")
print(expectation)

Random variable 'E(X)':
        E(X)
sample      
0      -0.75
1      -0.75
2      -0.75
3      -0.75
4      -0.75 

-0.75


### Conditional expectations

Define a $\sigma$-algebra $\mathcal{G}$ on $\Omega$ with atoms $A_0=\{0,1\}$, $A_1 = \{2\}$, $A_2 = \{3,4\}$. 

In [5]:
from sigalg.core import SigmaAlgebra

G = SigmaAlgebra(sample_space=Omega, name="G").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 2,
        4: 2,
    }
)

Compute the conditional expectation $E(X\mid \mathcal{G})$. Notice that $X$ is constant on the atom $A_0$ of $\mathcal{G}$, and hence $E(X\mid \mathcal{G})$ is also constant on this atom.

In [6]:
E(X,G)

Random variable 'E(X|G)':
          E(X|G)
sample          
0       1.000000
1       1.000000
2      -3.000000
3      -0.222222
4      -0.222222

### Testing properties of expectations

#### Conditional expectations are linear combinations

We noted in the definition that the conditional expectation may be expressed as a linear combination of the indicator functions of the atoms of the $\sigma$-algebra. In the next code cell, we test this. Notice that the output matches the output above.

In [7]:
I = RandomVariable.indicator_of

linear_combo = sum([E(X(A)).item() * I(A) for A in G.to_atoms()])

print(linear_combo.with_name("linear_combo"))

Random variable 'linear_combo':
        linear_combo
sample              
0           1.000000
1           1.000000
2          -3.000000
3          -0.222222
4          -0.222222


#### The law of iterated expectation

Given a sub-$\sigma$-algebra $\mathcal{H}$ of $\mathcal{G}$, the law of iterated expectation says that

$$
E(X \mid \mathcal{H}) = E(E(X \mid \mathcal{G}), \mathcal{H}).
$$

We verify this equality in the following code cell.

In [8]:
H = SigmaAlgebra(sample_space=Omega, name="H").from_dict(
    {
        0: 0,
        1: 0,
        2: 0,
        3: 1,
        4: 1,
    }
)

expectation = E(X, H)
iterated_expectation = E(E(X, G), H)

print(expectation, "\n")
print(iterated_expectation)

Random variable 'E(X|H)':
          E(X|H)
sample          
0      -1.181818
1      -1.181818
2      -1.181818
3      -0.222222
4      -0.222222 

Random variable 'E(E(X|G)|H)':
        E(E(X|G)|H)
sample             
0         -1.181818
1         -1.181818
2         -1.181818
3         -0.222222
4         -0.222222


#### Linearity

Given a second random variable $Y: \Omega \to \mathbb{R}$, linearity of expectation says that

$$
E(aX + bY \mid \mathcal{G}) = a E(X\mid \mathcal{G}) + b E(Y\mid \mathcal{G})
$$

for any scalars $a$ and $b$. We verify this equality in the following code cell.

In [9]:
Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: 1,
        1: -2,
        2: 3,
        3: -5,
        4: 1,
    }
)
Y.prob_measure = P

a = 2
b = -3
expectation = E(a * X + b * Y, G)
linear_combo = a * E(X, G) + b * E(Y, G)

print(expectation.with_name("expectation"), "\n")
print(linear_combo.with_name("linear_combo"))

Random variable 'expectation':
        expectation
sample             
0          4.400000
1          4.400000
2        -15.000000
3          6.555556
4          6.555556 

Random variable 'linear_combo':
        linear_combo
sample              
0           4.400000
1           4.400000
2         -15.000000
3           6.555556
4           6.555556


#### The pull-out property

Given a $\mathcal{G}$-measurable random variable $C: \Omega \to \mathbb{R}$, the pull-out property of expectation says that

$$
E(CX \mid \mathcal{G}) = C E(X\mid \mathcal{G}).
$$

We verify this equality in the next code cell.

In [10]:
C = RandomVariable(domain=Omega, name="C").from_dict(
    {
        0: 2,
        1: 2,
        2: -3,
        3: 1,
        4: 1,
    }
)
C.prob_measure = P

expectation = E(C * X, G)
pull_out = C * E(X, G)

print(expectation, "\n")
print(pull_out)


Random variable 'E((C*X)|G)':
        E((C*X)|G)
sample            
0         2.000000
1         2.000000
2         9.000000
3        -0.222222
4        -0.222222 

Random variable '(C*E(X|G))':
        (C*E(X|G))
sample            
0         2.000000
1         2.000000
2         9.000000
3        -0.222222
4        -0.222222
